# 03 - Model Training

**Purpose**: Train logistic regression models for each churn bucket.

**Spec Reference**: `specs/001-churn-prediction-model/spec.md`

## Model Configuration (per Constitution)

| Setting | Value | Rationale |
|---------|-------|----------|
| Algorithm | Logistic Regression | Interpretability (Principle III) |
| Class Weight | balanced | Handle class imbalance |
| CV Strategy | Stratified 5-Fold | Reproducibility (Principle II) |
| Performance Target | AUC-ROC ≥ 0.70 | Minimum acceptable performance |

## Prerequisites
- Run `02_feature_engineering.ipynb` first
- Features table populated in Lakehouse

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

# Project imports
import sys
sys.path.append('..')
from src.utils.config import get_config
from src.utils.logging import setup_logging, get_logger, ModelTrainingLogger
from src.models.training import train_logistic_regression, cross_validate_model
from src.models.evaluation import evaluate_model, calculate_metrics

# Setup
setup_logging()
logger = get_logger(__name__)
config = get_config()

print(f"Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Target AUC-ROC: >= {config.model.MIN_AUC_ROC}")

In [ ]:
# MLflow setup (optional - for experiment tracking)
try:
    import mlflow
    mlflow.set_experiment(config.model.EXPERIMENT_NAME)
    MLFLOW_AVAILABLE = True
    print(f"MLflow experiment: {config.model.EXPERIMENT_NAME}")
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not available - training without experiment tracking")

## 1. Load Features and Labels

⚠️ **ACTION REQUIRED**: Update Lakehouse paths below.

In [ ]:
# Load from Fabric Lakehouse
# Uncomment and update paths

# features_df = spark.read.format("delta").load("Tables/features").toPandas()
# print(f"Loaded {len(features_df)} feature records")

# For this notebook, assume labels are merged with features or loaded separately
# labels_df = features_df[['customer_id', 'label_30d', 'label_60d', 'label_90d', 'label_explicit']]

## 2. Prepare Training Data

In [ ]:
# Define feature columns (exclude IDs and labels)
# Uncomment when data loaded

# exclude_cols = ['customer_id', 'feature_id', 'feature_set_id', 'computed_date', 'created_at',
#                 'label_30d', 'label_60d', 'label_90d', 'label_explicit', 'tenure_bucket']
# feature_cols = [c for c in features_df.columns if c not in exclude_cols]
# print(f"Using {len(feature_cols)} features: {feature_cols}")

## 3. Train Models for Each Churn Bucket

Training separate models allows different feature importance per bucket.

In [ ]:
# Store results for each bucket
model_results = {}
churn_buckets = ['churn_30d', 'churn_60d', 'churn_90d', 'churn_explicit']

In [ ]:
# Train model for churn_30d
# Uncomment to run

# bucket = 'churn_30d'
# label_col = 'label_30d'
# 
# training_logger = ModelTrainingLogger(bucket)
# 
# # Prepare data
# X = features_df[feature_cols]
# y = features_df[label_col]
# 
# # Train/test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=config.model.RANDOM_STATE, stratify=y
# )
# 
# training_logger.start_training(len(X_train), X_train.shape[1])
# 
# # Cross-validation
# cv_results = cross_validate_model(X_train, y_train)
# print(f"CV AUC-ROC: {cv_results['cv_mean']:.4f} (+/- {cv_results['cv_std']:.4f})")
# 
# # Train final model
# model, scaler = train_logistic_regression(X_train, y_train)
# 
# # Evaluate
# X_test_scaled = scaler.transform(X_test)
# metrics = evaluate_model(model, X_test_scaled, y_test)
# 
# training_logger.log_metrics(metrics)
# training_logger.end_training(success=metrics['auc_roc'] >= config.model.MIN_AUC_ROC)
# 
# # Store results
# model_results[bucket] = {
#     'model': model,
#     'scaler': scaler,
#     'feature_names': feature_cols,
#     'metrics': metrics,
#     'cv_results': cv_results
# }
# 
# print(f"\n{bucket} Results:")
# print(f"  AUC-ROC: {metrics['auc_roc']:.4f}")
# print(f"  Precision: {metrics['precision']:.4f}")
# print(f"  Recall: {metrics['recall']:.4f}")
# print(f"  F1: {metrics['f1']:.4f}")

In [ ]:
# Repeat for other buckets (60d, 90d, explicit)
# Template - uncomment and modify label_col for each

# for bucket, label_col in [('churn_60d', 'label_60d'), 
#                           ('churn_90d', 'label_90d'),
#                           ('churn_explicit', 'label_explicit')]:
#     print(f"\n{'='*50}")
#     print(f"Training {bucket}")
#     print(f"{'='*50}")
#     
#     # ... (same code pattern as above)

## 4. Model Comparison

In [ ]:
# Compare all models
# from src.models.evaluation import compare_models

# if model_results:
#     comparison_df = compare_models(model_results)
#     print("Model Comparison:")
#     print(comparison_df.to_string(index=False))

## 5. Log to MLflow (if available)

In [ ]:
# Log models to MLflow
# if MLFLOW_AVAILABLE and model_results:
#     for bucket, results in model_results.items():
#         with mlflow.start_run(run_name=f"train_{bucket}"):
#             # Log parameters
#             mlflow.log_params({
#                 'bucket': bucket,
#                 'solver': config.model.SOLVER,
#                 'class_weight': config.model.CLASS_WEIGHT,
#                 'n_features': len(results['feature_names'])
#             })
#             
#             # Log metrics
#             mlflow.log_metrics(results['metrics'])
#             mlflow.log_metrics({
#                 'cv_mean_auc': results['cv_results']['cv_mean'],
#                 'cv_std_auc': results['cv_results']['cv_std']
#             })
#             
#             # Log model
#             mlflow.sklearn.log_model(results['model'], f"model_{bucket}")
#             
#             print(f"Logged {bucket} to MLflow")

## 6. Save Model Metrics to Lakehouse

In [ ]:
# Create model_metrics records
# import uuid
# import json

# metrics_records = []
# for bucket, results in model_results.items():
#     record = {
#         'metrics_id': str(uuid.uuid4()),
#         'model_version': f"v1_{datetime.now().strftime('%Y%m%d')}",
#         'churn_bucket': bucket,
#         'training_date': datetime.now().date(),
#         'auc_roc': results['metrics']['auc_roc'],
#         'auc_pr': results['metrics']['auc_pr'],
#         'accuracy': results['metrics']['accuracy'],
#         'precision_score': results['metrics']['precision'],
#         'recall_score': results['metrics']['recall'],
#         'f1_score': results['metrics']['f1'],
#         'cv_mean_auc': results['cv_results']['cv_mean'],
#         'cv_std_auc': results['cv_results']['cv_std'],
#         'hyperparameters_json': json.dumps({'solver': config.model.SOLVER, 'class_weight': config.model.CLASS_WEIGHT}),
#         'created_at': datetime.now()
#     }
#     metrics_records.append(record)

# metrics_df = pd.DataFrame(metrics_records)
# print(f"Prepared {len(metrics_df)} metrics records")

In [ ]:
# Save to Lakehouse
# spark_df = spark.createDataFrame(metrics_df)
# spark_df.write.format("delta").mode("append").save("Tables/model_metrics")
# print("Model metrics saved to Lakehouse!")

## 7. Save Models for Scoring

Save trained models for use in batch scoring notebook.

In [ ]:
# Save models locally for scoring notebook
# import joblib

# for bucket, results in model_results.items():
#     model_path = f"../models/{bucket}_model.joblib"
#     scaler_path = f"../models/{bucket}_scaler.joblib"
#     
#     joblib.dump(results['model'], model_path)
#     joblib.dump(results['scaler'], scaler_path)
#     print(f"Saved {bucket} model and scaler")

## Summary

| Bucket | AUC-ROC | Precision | Recall | F1 | Meets Target |
|--------|---------|-----------|--------|----|--------------|
| churn_30d | - | - | - | - | - |
| churn_60d | - | - | - | - | - |
| churn_90d | - | - | - | - | - |
| churn_explicit | - | - | - | - | - |

**Next Steps**:
1. Run `04_model_evaluation.ipynb` for detailed analysis
2. Run `06_feature_importance.ipynb` for coefficient analysis
3. Run `05_batch_scoring.ipynb` to generate predictions